In [29]:
# ============================================================================
#  Custom G-code generation for the serpentine strain sensor (FullControl)
#
#  The toolpath is defined analytically from a small set of geometric
#  parameters, transformed into G-code, and finally spliced into a pre-sliced
#  dogbone specimen so that the conductive sensor track is printed directly
#  onto the substrate.
#
#  Each numbered block below corresponds to one notebook cell.
# ============================================================================

In [30]:
# ============================== CELL 1 ======================================
#  Imports and global parameters
# ============================================================================

import fullcontrol as fc
import lab.fullcontrol as fclab
from math import tau
from pathlib import Path
import ctypes
from ctypes import wintypes
 
# --- Printer / material settings ---
printer_name = 'prusa_i3'   # FullControl printer profile
print_speed  = 15 * 60      # nominal print speed [mm/min]  (15 mm/s)
arc_speed    = 20 * 60      # print speed during arcs [mm/min]  (20 mm/s)
arc_flow     = 98           # flow during arcs [%] (reduces pressure, improves accuracy)
nozzle_temp  = 230          # nozzle temperature [degC]
bed_temp     = 70           # bed temperature [degC]
fan_percent  = 100          # part-cooling fan [%]
 
# --- Extrusion geometry ---
EW = 0.4   # extrusion width [mm]
EH = 0.2   # extrusion height / layer height [mm]

In [31]:
# ============================== BLOCK 2 =====================================
#  Sensor configuration
# ============================================================================

# line_depth sets how far the conductive track is embedded into the substrate.
#   0.000  -> nozzle sits one layer height (EH) above the substrate top surface
#   0.050  -> nozzle is pushed closer, reducing the gap from 0.20 mm to 0.15 mm
line_depth = 0.045
 
# One dictionary per sensor. Append further dictionaries to print several
# sensors in one run; a travel move is inserted between consecutive sensors.
sensor_configs = [
    {   # Serpentine strain sensor
        'segments':  64,       # arc facet count (higher gives smoother arcs)
        'amplitude': 6,        # serpentine peak-to-peak height [mm]
        'radius':    0.245,    # serpentine turn radius [mm]  (510 um turn diameter)
        'units':     10,       # number of serpentine periods
        'line_depth': line_depth,
    },
]
 
# --- Substrate placement (bed coordinates of the sensor centre) ---
substrate_x      = 180   # x-position of sensor centre on the bed [mm]
substrate_y      = 180   # y-position of sensor centre on the bed [mm]
substrate_height = 1     # substrate thickness [mm]
 
# --- Output naming ---
design_name = f'PLA_R245,{line_depth:.3f}'
output_dir  = 'Sensor_gcodes'

In [32]:
# ============================== BLOCK 3 =====================================
#  Toolpath generation
# ============================================================================

all_steps  = []   # extrusion path (may be copied or transformed later)
init_steps = []   # setup moves that must not be duplicated
 
for sensor_idx, config in enumerate(sensor_configs):
    segments   = config['segments']
    amplitude  = config['amplitude']
    radius     = config['radius']
    units      = config['units']
    line_depth = config['line_depth']
 
    # Contact-pad and line dimensions
    pad_depth  = 0            # extra depth of the contact pads (added to line depth)
    pad_width  = amplitude    # pads are as wide as the serpentine is tall
    line_width = EW
 
    # Sensor origin chosen so that its centre is at (0, 0)
    sensor_length = 2*pad_width + 2*radius + 2 + 2*units*2*radius
    x_0 = -sensor_length/2 + pad_width
    y_0 = -amplitude/2
    z_0 = 0
 
    # Translate the origin onto the substrate on the print bed
    x_1 = x_0 + substrate_x
    y_1 = y_0 + substrate_y
    z_1 = z_0 + substrate_height + EH - line_depth
    z_start = z_1 + 2         # safe approach height above the start point
 
    # Forced approach move, added only once before the first sensor
    if sensor_idx == 0:
        init_steps.append(fc.ManualGcode(text=f"G0 X{x_1} Y{y_1} Z{z_start}; Force starting point"))
        init_steps.append(fc.ManualGcode(text=f"G0 X{x_1} Y{y_1} Z{z_1}; Force starting point"))
 
    steps = []
 
    # Move to the sensor start without extruding, then enable the extruder
    steps.append(fc.Extruder(on=False))
    steps.append(fc.Point(x=x_1, y=y_1, z=z_1))   # reference start point
    steps.append(fc.Extruder(on=True))
 
    # Relative-coordinate helper: moves are relative to the last point in `steps`
    R = fclab.setup_r(steps)
 
    # --- Start contact pad (square-wave infill) ---
    steps.extend(fc.squarewaveXY(R(0, 0, -pad_depth), fc.Vector(x=0, y=1),
                                 amplitude, line_width,
                                 int(amplitude/(2*line_width)), True, False))
    steps.append(R(0, line_width, 0))   # finish the final half period
    steps.append(R(amplitude, 0, 0))
 
    # --- Transition into the serpentine ---
    # Arcs are printed slower and at reduced flow to limit pressure build-up and
    # improve dimensional accuracy; nominal speed and flow are restored afterwards.
    steps.append(R(1, 0, pad_depth))    # step off the pad
    steps.append(fc.ManualGcode(text=f"G1 F{arc_speed}"))   # arc speed
    steps.append(fc.ManualGcode(text=f"M221 S{arc_flow}"))  # arc flow
    steps.extend(fc.arcXY(R(0, -radius, 0), radius, 0.25*tau, -0.25*tau, segments))
    steps.append(fc.ManualGcode(text=f"G1 F{print_speed}")) # restore nominal speed
    steps.append(fc.ManualGcode(text="M221 S100"))          # restore 100 % flow
 
    # --- Serpentine body ---
    for _ in range(units):
        steps.append(R(0, -(amplitude - 2*radius), 0))
        steps.append(fc.ManualGcode(text=f"G1 F{arc_speed}"))
        steps.append(fc.ManualGcode(text=f"M221 S{arc_flow}"))
        steps.extend(fc.arcXY(R(radius, 0, 0), radius, -0.5*tau,  0.5*tau, segments))
        steps.append(fc.ManualGcode(text=f"G1 F{print_speed}"))
        steps.append(fc.ManualGcode(text="M221 S100"))
 
        steps.append(R(0,  (amplitude - 2*radius), 0))
        steps.append(fc.ManualGcode(text=f"G1 F{arc_speed}"))
        steps.append(fc.ManualGcode(text=f"M221 S{arc_flow}"))
        steps.extend(fc.arcXY(R(radius, 0, 0), radius, -0.5*tau, -0.5*tau, segments))
        steps.append(fc.ManualGcode(text=f"G1 F{print_speed}"))
        steps.append(fc.ManualGcode(text="M221 S100"))
 
    # --- Transition out of the serpentine ---
    steps.append(R(0, -(amplitude - 2*radius), 0))
    steps.append(fc.ManualGcode(text=f"G1 F{arc_speed}"))   # arc speed
    steps.append(fc.ManualGcode(text=f"M221 S{arc_flow}"))  # arc flow
    steps.extend(fc.arcXY(R(radius, 0, 0), radius, 0.5*tau, 0.25*tau, segments))
    steps.append(fc.ManualGcode(text=f"G1 F{print_speed}")) # restore nominal speed
    steps.append(fc.ManualGcode(text="M221 S100"))          # restore 100 % flow
    steps.append(R(1, 0, -pad_depth))   # step off the path
 
    # --- End contact pad (square-wave infill) ---
    steps.extend(fc.squarewaveXY(R(0, 0, 0), fc.Vector(x=0, y=1),
                                 -amplitude, line_width,
                                 int(amplitude/(2*line_width)), True, False))
    steps.append(R(0, line_width, 0))   # finish the final half period
    steps.append(R(-amplitude, 0, 0))
 
    steps.append(fc.Extruder(on=False))
 
    # Travel move to the next sensor, or a final lift after the last one
    if sensor_idx < len(sensor_configs) - 1:
        steps.append(R(0, 1, 1))
        steps.append(R(-15, 0, 1))
        steps.append(fc.Extruder(on=True))
    else:
        steps.append(R(0, 2, 1))
        steps.append(R(-20, 0, 4))
        steps.append(R(0, 10, 0))
        steps.append(fc.Extruder(on=True))
 
    all_steps.extend(steps)
 
# Complete step list: setup moves first, then the sensor path(s)
steps = init_steps + all_steps

In [33]:
# ============================== BLOCK 4 =====================================
#  G-code transformation and export
# ============================================================================

# Full G-code including the printer start/end sequence and a primer
gcode_controls = fc.GcodeControls(
    printer_name=printer_name,
    initialization_data={
        'primer':           'front_lines_then_y',
        'print_speed':      print_speed,
        'nozzle_temp':      nozzle_temp,
        'bed_temp':         bed_temp,
        'fan_percent':      fan_percent,
        'extrusion_width':  EW,
        'extrusion_height': EH,
    })
 
# Bare G-code without a start/end sequence, for embedding into an existing print
gcode_controls_bare = fc.GcodeControls(
    printer_name='generic',
    initialization_data={
        'print_speed':      print_speed,
        'extrusion_width':  EW,
        'extrusion_height': EH,
    })
 
gcode      = fc.transform(steps, 'gcode', gcode_controls)
gcode_bare = fc.transform(steps, 'gcode', gcode_controls_bare)
 
# Optional design preview
fc.transform(steps, 'plot', fc.PlotControls(style="line", zoom=0.7))

In [34]:
# ============================== BLOCK 5 =====================================
#  Clipboard helper (Windows)
# ============================================================================

def copy_file_to_windows_clipboard(file_path: Path) -> None:
    """Copy a file reference to the Windows clipboard so it can be pasted in Explorer."""
    if not str(file_path).strip():
        raise ValueError("file_path cannot be empty")
 
    resolved = Path(file_path).resolve()
    if not resolved.exists():
        raise FileNotFoundError(f"File does not exist: {resolved}")
 
    user32   = ctypes.windll.user32
    kernel32 = ctypes.windll.kernel32
 
    user32.OpenClipboard.argtypes    = [wintypes.HWND]
    user32.OpenClipboard.restype     = wintypes.BOOL
    user32.EmptyClipboard.argtypes   = []
    user32.EmptyClipboard.restype    = wintypes.BOOL
    user32.SetClipboardData.argtypes = [wintypes.UINT, wintypes.HANDLE]
    user32.SetClipboardData.restype  = wintypes.HANDLE
    user32.CloseClipboard.argtypes   = []
    user32.CloseClipboard.restype    = wintypes.BOOL
 
    kernel32.GlobalAlloc.argtypes  = [wintypes.UINT, ctypes.c_size_t]
    kernel32.GlobalAlloc.restype   = wintypes.HGLOBAL
    kernel32.GlobalLock.argtypes   = [wintypes.HGLOBAL]
    kernel32.GlobalLock.restype    = wintypes.LPVOID
    kernel32.GlobalUnlock.argtypes = [wintypes.HGLOBAL]
    kernel32.GlobalUnlock.restype  = wintypes.BOOL
    kernel32.GlobalFree.argtypes   = [wintypes.HGLOBAL]
    kernel32.GlobalFree.restype    = wintypes.HGLOBAL
 
    if not user32.OpenClipboard(None):
        raise ctypes.WinError()
 
    CF_HDROP = 15
    GHND     = 0x0042
 
    class POINT(ctypes.Structure):
        _fields_ = [("x", wintypes.LONG), ("y", wintypes.LONG)]
 
    class DROPFILES(ctypes.Structure):
        _fields_ = [
            ("pFiles", wintypes.DWORD),
            ("pt", POINT),
            ("fNC", wintypes.BOOL),
            ("fWide", wintypes.BOOL),
        ]
 
    # CF_HDROP expects a double-null-terminated UTF-16 string list
    file_list_data = (str(resolved) + "\0\0").encode("utf-16le")
    total_size = ctypes.sizeof(DROPFILES) + len(file_list_data)
 
    hglobal = kernel32.GlobalAlloc(GHND, total_size)
    if not hglobal:
        user32.CloseClipboard()
        raise ctypes.WinError()
 
    pcontents = kernel32.GlobalLock(hglobal)
    if not pcontents:
        kernel32.GlobalFree(hglobal)
        user32.CloseClipboard()
        raise ctypes.WinError()
 
    try:
        dropfiles = DROPFILES()
        dropfiles.pFiles = ctypes.sizeof(DROPFILES)
        dropfiles.pt     = POINT(0, 0)
        dropfiles.fNC    = False
        dropfiles.fWide  = True
 
        ctypes.memmove(pcontents, ctypes.byref(dropfiles), ctypes.sizeof(DROPFILES))
        ctypes.memmove(pcontents + ctypes.sizeof(DROPFILES), file_list_data, len(file_list_data))
    finally:
        kernel32.GlobalUnlock(hglobal)
 
    try:
        if not user32.EmptyClipboard():
            raise ctypes.WinError()
        if not user32.SetClipboardData(CF_HDROP, hglobal):
            raise ctypes.WinError()
        # On success, Windows owns hglobal; do not free it.
        hglobal = None
    finally:
        if hglobal:
            kernel32.GlobalFree(hglobal)
        user32.CloseClipboard()

In [ ]:
# ============================== BLOCK 6 =====================================
#  Embed the sensor path into the sliced substrate print and export
# ============================================================================

# Base print: a sliced dogbone specimen (TPU substrate + PLA sensing) from the slicer
with open('Dogbone_single_TPU_PLA.gcode', 'r') as ofile:
    original = ofile.readlines()
 
layer = original.index(';Z:1.2\n')   # first line of the target layer (Z = 1.2 mm)
 
# All wipe-end markers within and after the target layer
inds = [i + layer for i, x in enumerate(original[layer:]) if x == ';WIPE_END\n']
 
# Comment markers surrounding the injected FullControl block
FC_start = ['\n', 'M486 S0\n', ';FC start\n', '\n']
FC_end   = ['\n', ';FC end\n', '\n']
 
# Replace the region between the first and last wipe with the bare sensor G-code.
# gcode_bare[229:] drops the FullControl header and keeps only the motion commands.
augmented = (original[:inds[0] + 1]
             + FC_start + [gcode_bare[229:]] + FC_end
             + original[inds[-1] + 1:])
 
# Write the combined G-code
output_path = Path(output_dir) / f'{design_name}.gcode'
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as write:
    write.writelines(augmented)
 
# Copy the exported file to the Windows clipboard for quick pasting
copy_file_to_windows_clipboard(output_path)
print(f"Saved and copied to clipboard: {output_path.resolve()}")

Saved and copied to clipboard: C:\Users\lukas\OneDrive - Danmarks Tekniske Universitet\Bjørnebanden.Semester\Master\Initial sensor tests\Sensor_gcodes\PLA_R245,0.045.gcode
